# Détection d'Anomalies de Sertissage avec One-Class SVM
Ce notebook permet d'entraîner, de valider (optimisation des hyperparamètres `nu` et `gamma`), de tester et de sauvegarder un modèle **One-Class SVM** sur le dataset Zenodo (`crimp_force_curves_dataset.pkl`).

In [1]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import OneClassSVM
from sklearn.metrics import classification_report, accuracy_score, f1_score, confusion_matrix

## 1. Chargement du dataset de courbes de sertissage

In [2]:
print("Chargement du dataset...")
df = pd.read_pickle("crimp_force_curves_dataset.pkl")

# Extraction des courbes brutes et des étiquettes binaires (0 = OK, 1 = NOK / Anomalie)
X_raw = df['Force_curve_raw'].tolist()
y_binary = df['Binary_label_encoded'].values

print(f"Nombre total de courbes : {len(X_raw)}")
print(f"Répartition (0=OK, 1=NOK) : {np.bincount(y_binary)}")

Chargement du dataset...
Nombre total de courbes : 2439
Répartition (0=OK, 1=NOK) : [1628  811]


## 2. Extraction des caractéristiques (Feature Engineering)

In [3]:
print("Extraction des caractéristiques des courbes...")
features = []
for curve in X_raw:
    arr = np.array(curve)
    feat = [
        np.max(arr),       # Force maximale atteinte
        np.min(arr),       # Force minimale
        np.mean(arr),      # Force moyenne
        np.std(arr),       # Écart-type (variation du signal)
        np.trapz(arr),     # Aire sous la courbe (énergie de sertissage)
        np.argmax(arr)     # Position temporelle du pic de force
    ]
    features.append(feat)

X = np.array(features)

Extraction des caractéristiques des courbes...


## 3. Préparation des ensembles : Entraînement (OK uniquement), Validation et Test
* **Entraînement (`X_train_ok`)** : Contient **uniquement** des courbes conformes (`OK`, label = `0`), car One-Class SVM apprend la frontière de la normalité.
* **Validation (`X_val`, `y_val`)** : Contient un mélange de courbes `OK` et `NOK` pour ajuster les hyperparamètres (`nu` et `gamma`).
* **Test (`X_test`, `y_test`)** : Contient un mélange de courbes `OK` et `NOK` pour l'évaluation finale.

In [4]:
# Séparation des courbes normales (OK = 0) et défectueuses (NOK = 1)
X_ok = X[y_binary == 0]
y_ok = y_binary[y_binary == 0]

X_nok = X[y_binary == 1]
y_nok = y_binary[y_binary == 1]

# 1. Découpage des courbes OK : 60% Train, 20% Validation, 20% Test
X_train_ok, X_temp_ok, y_train_ok, y_temp_ok = train_test_split(
    X_ok, y_ok, test_size=0.40, random_state=42
)
X_val_ok, X_test_ok, y_val_ok, y_test_ok = train_test_split(
    X_temp_ok, y_temp_ok, test_size=0.50, random_state=42
)

# 2. Découpage des courbes NOK : 50% Validation, 50% Test (aucune dans l'entraînement)
X_val_nok, X_test_nok, y_val_nok, y_test_nok = train_test_split(
    X_nok, y_nok, test_size=0.50, random_state=42
)

# 3. Assemblage final des ensembles de Validation et de Test
X_val = np.vstack((X_val_ok, X_val_nok))
y_val = np.concatenate((y_val_ok, y_val_nok))

X_test = np.vstack((X_test_ok, X_test_nok))
y_test = np.concatenate((y_test_ok, y_test_nok))

print("--- Répartition des données ---")
print(f"Entraînement (OK uniquement) : {len(X_train_ok)} courbes")
print(f"Validation (OK + NOK)        : {len(X_val)} courbes (OK: {len(y_val_ok)}, NOK: {len(y_val_nok)})")
print(f"Test (OK + NOK)              : {len(X_test)} courbes (OK: {len(y_test_ok)}, NOK: {len(y_test_nok)})")

--- Répartition des données ---
Entraînement (OK uniquement) : 976 courbes
Validation (OK + NOK)        : 731 courbes (OK: 326, NOK: 405)
Test (OK + NOK)              : 732 courbes (OK: 326, NOK: 406)


## 4. Normalisation des données (Obligatoire pour SVM)
Le `StandardScaler` est ajusté **uniquement** sur les données d'entraînement (`X_train_ok`) puis appliqué à la validation et au test.

In [5]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_ok)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

## 5. Entraînement et Validation (Optimisation de `nu` et `gamma`)
**Note sur les prédictions de One-Class SVM :**
* Le modèle retourne **`1`** pour une pièce normale (`OK` -> converti en `0`).
* Le modèle retourne **`-1`** pour une anomalie (`NOK` -> converti en `1`).

In [6]:
nu_values = [0.01, 0.03, 0.05, 0.1, 0.15]
gamma_values = ['scale', 'auto', 0.01, 0.1, 1.0]

best_f1 = -1
best_params = {}
best_ocsvm = None

print("Recherche des meilleurs hyperparamètres sur l'ensemble de validation...")

for nu in nu_values:
    for gamma in gamma_values:
        # Initialisation et entraînement sur X_train_scaled (OK uniquement)
        model = OneClassSVM(kernel='rbf', nu=nu, gamma=gamma)
        model.fit(X_train_scaled)
        
        # Prédiction sur la validation
        val_preds_raw = model.predict(X_val_scaled)
        # Conversion : 1 -> 0 (OK), -1 -> 1 (NOK)
        val_preds = np.where(val_preds_raw == 1, 0, 1)
        
        # Évaluation avec le F1-score (adapté aux classes déséquilibrées)
        score = f1_score(y_val, val_preds)
        
        if score > best_f1:
            best_f1 = score
            best_params = {'nu': nu, 'gamma': gamma}
            best_ocsvm = model

print(f"\nMeilleurs paramètres trouvés : {best_params}")
print(f"Meilleur F1-Score sur la validation : {best_f1:.4f}")

Recherche des meilleurs hyperparamètres sur l'ensemble de validation...

Meilleurs paramètres trouvés : {'nu': 0.15, 'gamma': 1.0}
Meilleur F1-Score sur la validation : 0.8951


## 6. Évaluation finale sur l'ensemble de TEST

In [7]:
test_preds_raw = best_ocsvm.predict(X_test_scaled)
y_test_pred = np.where(test_preds_raw == 1, 0, 1)

acc_test = accuracy_score(y_test, y_test_pred)
print(f"[Test] Précision finale (Accuracy) : {acc_test:.4f}\n")

print("Matrice de confusion (Lignes = Réel, Colonnes = Prédit) :")
print(confusion_matrix(y_test, y_test_pred))

print("\nRapport de classification détaillé (0 = OK, 1 = NOK) :")
print(classification_report(y_test, y_test_pred, target_names=['OK (0)', 'NOK (1)']))

[Test] Précision finale (Accuracy) : 0.8429

Matrice de confusion (Lignes = Réel, Colonnes = Prédit) :
[[265  61]
 [ 54 352]]

Rapport de classification détaillé (0 = OK, 1 = NOK) :
              precision    recall  f1-score   support

      OK (0)       0.83      0.81      0.82       326
     NOK (1)       0.85      0.87      0.86       406

    accuracy                           0.84       732
   macro avg       0.84      0.84      0.84       732
weighted avg       0.84      0.84      0.84       732



## 7. Sauvegarde du modèle et du StandardScaler
Il est indispensable de sauvegarder **le scaler en même temps que le modèle** pour pouvoir normaliser les futures courbes en production exactement de la même manière.

In [8]:
artifact = {
    'model': best_ocsvm,
    'scaler': scaler,
    'best_params': best_params
}

model_filename = "crimp_one_class_svm_model.pkl"
joblib.dump(artifact, model_filename)
print(f"Modèle et scaler sauvegardés avec succès sous : {model_filename}")

Modèle et scaler sauvegardés avec succès sous : crimp_one_class_svm_model.pkl
